In [ ]:
# ==========================================
# 1️⃣ Import des librairies
# ==========================================
import h2o
from h2o.automl import H2OAutoML
from h2o.estimators.deeplearning import H2ODeepLearningEstimator

# ==========================================
# 2️⃣ Démarrer H2O
# ==========================================
h2o.init()

# ==========================================
# 3️⃣ Charger les données
# ==========================================
data = h2o.import_file("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

# ==========================================
# 4️⃣ Définir variables
# ==========================================
y = "survived"
X = data.columns
X.remove(y)

# Convertir en classification
data[y] = data[y].asfactor()

# ==========================================
# 5️⃣ Nettoyage SIMPLE (corrigé ✅)
# ==========================================
# ⚠️ PAS de data[col] = ...
for col in X:
    if data[col].isnumeric()[0]:
        data[col].impute(method="median")

# ==========================================
# 6️⃣ Split train/test
# ==========================================
train, test = data.split_frame(ratios=[0.8], seed=42)

# ==========================================
# 7️⃣ NAS manuel (Deep Learning)
# ==========================================
architectures = [
    [32, 32],
    [64, 32],
    [128, 64, 32],
    [50, 50, 50]
]

results = []

for arch in architectures:
    print(f"\n🔍 Test architecture : {arch}")

    model = H2ODeepLearningEstimator(
        hidden=arch,
        epochs=20,
        activation="Rectifier",
        seed=42,
        stopping_rounds=5  # évite overfitting
    )

    model.train(x=X, y=y, training_frame=train)

    perf = model.model_performance(test_data=test)
    auc = perf.auc()

    print(f"AUC : {auc}")

    results.append((arch, model, auc))

# ==========================================
# 8️⃣ Meilleur modèle NAS
# ==========================================
best_arch, best_model, best_auc = max(results, key=lambda x: x[2])

print("\n🏆 Meilleure architecture :", best_arch)
print("🏆 Meilleure AUC :", best_auc)

# ==========================================
# 9️⃣ NAS automatique (AutoML)
# ==========================================
aml = H2OAutoML(
    max_models=10,
    max_runtime_secs=300,
    seed=42
)

aml.train(x=X, y=y, training_frame=train)

# Leaderboard
print("\n📊 Leaderboard :")
print(aml.leaderboard.head())

# Meilleur modèle
leader = aml.leader

print("\n🏆 Meilleur modèle AutoML :")
print(leader)

# Performance
perf = leader.model_performance(test_data=test)
print("\n📈 Performance test :")
print(perf)

# ==========================================
# 🔟 Sauvegarde modèle
# ==========================================
path = h2o.save_model(leader, path="./models", force=True)
print("\n💾 Modèle sauvegardé :", path)

# ==========================================
# 1️⃣1️⃣ Chargement modèle
# ==========================================
loaded_model = h2o.load_model(path)
print("\n✅ Modèle rechargé :", loaded_model)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,3 mins 44 secs
H2O_cluster_timezone:,America/Toronto
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month
H2O_cluster_name:,H2O_from_python_User_9jfpon
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.914 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%

🔍 Test architecture : [32, 32]
deeplearning Model Build progress: |█████████████████████████████████████████████| (done) 100%
AUC : 1.0

🔍 Test architecture : [64, 32]
deeplearning Model Build progress: |█████████████████████████████████████████████| (done) 100%
AUC : 1.0

🔍 Test architecture : [128, 64, 32]
deeplearning Model Build progress: |█████████████████████████████████████████████| (done) 100%
AUC : 1.0

🔍 Test architecture : [50, 50, 50]
deeplearning Model Build progress: |█████████████████████████████████████████████| (done) 100%
AUC : 1.0

🏆 Meilleure architecture : [32, 32]
🏆 Meilleure AUC : 1.0
AutoML progress: |█
15:16:14.26: AutoML: XGBoost is not available; skipping it.

████████